# Feature importance — ml_v2 prod models

Loads the latest `*_prod_*.joblib` from `ml_v2/models/{apartment|house}/` and prints
`feature_importances_` as percentages.

If the notebook kernel is Python 3.14 (XGBoost unpickling bug), loads the model via
`py -3.12` automatically.

In [2]:
from pathlib import Path

import pandas as pd

ML_V2_DIR = Path.cwd().resolve()
while ML_V2_DIR != ML_V2_DIR.parent and not (ML_V2_DIR / "repif_ml_v2").is_dir():
    ML_V2_DIR = ML_V2_DIR.parent

MODELS_DIR = ML_V2_DIR / "models"


def latest_prod_model(property_name: str) -> Path:
    folder = MODELS_DIR / property_name
    candidates = sorted(
        folder.glob(f"{property_name}_prod_*.joblib"),
        key=lambda p: p.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(f"No prod model in {folder}")
    return candidates[-1]


APARTMENT_MODEL = latest_prod_model("apartment")
HOUSE_MODEL = latest_prod_model("house")

print(APARTMENT_MODEL)
print(HOUSE_MODEL)

C:\Users\thoma\repos_ahiru\repif\ml_v2\models\apartment\apartment_prod_20260723_144015.joblib
C:\Users\thoma\repos_ahiru\repif\ml_v2\models\house\house_prod_20260723_162418.joblib


In [3]:
import json
import subprocess

import joblib


def _feature_importance_via_subprocess(model_path: Path) -> pd.DataFrame:
    code = """
import json, sys, joblib
import pandas as pd
model = joblib.load(sys.argv[1])
s = pd.Series(model.feature_importances_, index=model.feature_names_in_)
print(json.dumps({k: float(v) * 100 for k, v in s.items()}))
"""
    for launcher in (["py", "-3.12"], ["python3.12"], ["python3"]):
        try:
            proc = subprocess.run(
                [*launcher, "-c", code, str(model_path)],
                capture_output=True,
                text=True,
                check=True,
            )
            payload = json.loads(proc.stdout.strip())
            return pd.DataFrame.from_dict(
                payload, orient="index", columns=["importance"]
            ).sort_values("importance", ascending=False)
        except (FileNotFoundError, subprocess.CalledProcessError):
            continue
    raise RuntimeError(
        "Could not load model: install Python 3.12 (Windows: `py -3.12`) "
        "or run this notebook with a 3.12 kernel."
    )


def feature_importance_df(model_path: Path) -> pd.DataFrame:
    try:
        model = joblib.load(model_path)
    except Exception as exc:
        if type(exc).__name__ != "XGBoostError" and "corrupted" not in str(exc).lower():
            raise
        print(f"Fallback subprocess load for {model_path.name}")
        return _feature_importance_via_subprocess(model_path)

    series = pd.Series(
        model.feature_importances_,
        index=model.feature_names_in_,
    ).sort_values(ascending=False)
    out = pd.DataFrame(series, columns=["importance"])
    out["importance"] = out["importance"] * 100
    return out

In [4]:
pd_apartment = feature_importance_df(APARTMENT_MODEL)
print(APARTMENT_MODEL.name)
pd_apartment

Fallback subprocess load for apartment_prod_20260723_144015.joblib
apartment_prod_20260723_144015.joblib


,importance
commune_price_m2_median,39.903554
sbati,26.260686
apartement_rooms,19.314681
l_codinsee,6.642356
lon,2.820382
lat,1.991545
nblocdep,1.267726
annee_construction,1.143993
dpe_median,0.395389
sterr,0.259683


In [5]:
pd_house = feature_importance_df(HOUSE_MODEL)
print(HOUSE_MODEL.name)
pd_house

Fallback subprocess load for house_prod_20260723_162418.joblib
house_prod_20260723_162418.joblib


,importance
commune_price_m2_median,43.532610
sbati,28.751266
home_rooms,11.303202
l_codinsee,3.654962
log_sterr,2.939365
dpe_median,2.309044
lat,2.137583
lon,2.074742
annee_construction,1.907595
nblocdep,1.389635
